# Exponential Lyapunov Experiments

This notebook isolates the exponential Lyapunov experiments for the 1D MALA light-tail study. The emphasis is again on total drift, regional drift, proposal probabilities, accepted probabilities, and `R(x)`.

In [ ]:
import importlib.util
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

candidate_roots = [Path.cwd(), Path.cwd().parent]
repo_root = next(root for root in candidate_roots if (root / "src" / "mala_1d.py").exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

module_path = repo_root / "src" / "mala_1d.py"
spec = importlib.util.spec_from_file_location("mala_1d", module_path)
mala_1d = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mala_1d)

In [ ]:
x_grid = np.linspace(0.5, 6.0, 30)
n_samples = 20_000
seed = 12345
eps = 0.1


def run_and_plot_experiment(p, h, s, beta=1.0):
    lyapunov = mala_1d.build_lyapunov(
        lyapunov_type="exp",
        p=p,
        s=s,
        beta=beta,
    )

    result = mala_1d.run_experiment(
        x_grid=x_grid,
        h=h,
        p=p,
        n_samples=n_samples,
        seed=seed,
        V_func=lyapunov["V_func"],
        log_V_func=lyapunov["log_V_func"],
        eps=eps,
    )

    mala_1d.plot_experiment(
        result=result,
        p=p,
        h=h,
        lyapunov_label=lyapunov["label"],
        eps=eps,
        output_path=None,
    )

    return result, lyapunov

## Experiment 1: `p=2`, `h=0.05`, `V(x)=\exp(0.02|x|)`

In [ ]:
result_1, lyapunov_1 = run_and_plot_experiment(p=2, h=0.05, s=0.02)

This mild exponential weight tests whether the inward drift picture survives beyond polynomial Lyapunov functions. The main comparison is between the negative `A1` contribution and the rapidly shrinking accepted outward mass.

## Experiment 2: `p=2`, `h=0.05`, `V(x)=\exp(0.05|x|)`

In [ ]:
result_2, lyapunov_2 = run_and_plot_experiment(p=2, h=0.05, s=0.05)

With the stronger exponential rate, large excursions are weighted more heavily. The regional and acceptance panels help confirm whether the main obstruction is still proposal overshoot rather than a genuinely large outward accepted contribution.

## Experiment 3: `p=3`, `h=0.01`, `V(x)=\exp(0.02|x|)`

In [ ]:
result_3, lyapunov_3 = run_and_plot_experiment(p=3, h=0.01, s=0.02)

This is the clearest exponential example of early rejection collapse under stronger tails. The `R(x)` panel and accepted-probability panel together show how the deterministic drift scale starts to outpace the range where proposals remain locally acceptable.

## Summary

Across the exponential Lyapunov experiments, the same qualitative pattern persists:

- the negative drift is still driven primarily by `A1`,
- `A3` remains negligible,
- stronger tails push the system into rejection collapse sooner,
- and the practical obstruction is proposal overshoot followed by loss of accepted mass.